In [1]:
%cd ..

d:\github\ssd


In [2]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from model.ssd import SSD
from loss import MultiBoxLoss
from config import load_config_from_yaml
from training.dataloader import get_voc_dataloaders

In [3]:
dataloaders = get_voc_dataloaders(
    batch_size=2,
)

In [5]:
for images, targets in train_loader:
    break

c:\Users\LONG\anaconda3\envs\dzeus\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
targets[2]

tensor([[ 0.4956,  0.2020,  0.6696,  0.6288, 14.0000],
        [ 0.2355,  0.2803,  0.8243,  0.8308, 12.0000]])

In [4]:
model_config = load_config_from_yaml(
    config_type='model',
    file_path='./configs/model/ssd300.yaml'
)

train_config = load_config_from_yaml(
    config_type='train',
    file_path='./configs/train.yaml'
)

model = SSD(model_config)
# state_dict = torch.load('./weights/best.pt')
# model.load_state_dict(state_dict)

In [5]:
from torch.optim import Adam
from training.trainer import Trainer
from training.checkpointer import Checkpointer
from training.scheduler import ExponentialDecayScheduler


criterion = MultiBoxLoss(
    num_classes=21,
    iou_threshold=0.45,
    hard_negative_ratio=3,
    default_boxes=model.default_boxes,
    box_variances=model_config.box_variances,
    background_label=0
)

checkpointer = Checkpointer(
    checkpoint_dir="./weights"
)

optimizer = Adam(
    params=model.parameters(),
    lr=1e-5,
    betas=(0.9, 0.98),
    eps=1e-09,
)

scheduler = ExponentialDecayScheduler(
    optimizer=optimizer,
    gamma=0.9
)

train_config.device = 'cpu'

trainer = Trainer(
    config=train_config,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    checkpointer=checkpointer,
    scheduler=scheduler
)

In [6]:
trainer.fit(
    train_loader=dataloaders['train'],
    val_loader=dataloaders['validation']
)

Training Epoch 01:   0%|          | 0/4109 [00:00<?, ?it/s]c:\Users\LONG\anaconda3\envs\dzeus\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training Epoch 01:   1%|          | 39/4109 [01:51<3:13:33,  2.85s/it, loc_loss=3.5633, conf_loss=12.4636, lr=1.000000e-05]


KeyboardInterrupt: 

In [16]:
loc, conf = model(images)

In [ ]:
total = loc_loss + conf_loss
total.backward()

In [ ]:
image_numpy = image.cpu().numpy()
image_numpy = image_numpy.transpose(1, 2, 0)
rgb_image = cv2.cvtColor(image_numpy, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,10))
plt.imshow(rgb_image)
plt.show()

In [ ]:
from inference import ssd_inference, visualize_bboxes

In [ ]:
results = ssd_inference(
    loc=loc,
    conf=conf,
    default_boxes=model.default_boxes,
    num_classes=model_config.num_classes,
    background_label=0,
    top_k=100,
    conf_threshold=0.6,
    iou_threshold=0.4,
    box_variances=model_config.box_variances,
)

visualize_bboxes(
    rgb_image=rgb_image,
    bboxes=results,
    labels=[
        'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
        'bus', 'car', 'cat', 'chair', 'cow', 'diningtable',
        'dog', 'horse', 'motorbike', 'person', 'pottedplant',
        'sheep', 'sofa', 'train', 'tvmonitor'
    ],
    conf_threshold=0.6,
)